## Logistic Regression Baseline

We start with Logistic Regression as a simple and interpretable baseline for predicting SECOM labels.  
Since the main goal is to detect the `1` class correctly, we focus on recall for `1` rather than overall accuracy.  
A high recall for `1` means fewer false negatives, which is important when missing a defective sample is more costly than flagging an extra one.  
We also use `class_weight='balanced'` to handle the class imbalance in the dataset.

In [4]:
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, confusion_matrix, classification_report

save_dir = "prepared_pca_datasets"

pca_datasets = {}
for thr in [0.80, 0.90, 0.95]:
    thr_name = int(thr * 100)

    X_train_pca = joblib.load(os.path.join(save_dir, f"X_train_pca_{thr_name}.pkl"))
    X_test_pca  = joblib.load(os.path.join(save_dir, f"X_test_pca_{thr_name}.pkl"))
    y_train     = joblib.load(os.path.join(save_dir, f"y_train_{thr_name}.pkl"))
    y_test      = joblib.load(os.path.join(save_dir, f"y_test_{thr_name}.pkl"))

    pca_datasets[thr] = {
        "X_train": X_train_pca,
        "X_test": X_test_pca,
        "y_train": y_train,
        "y_test": y_test
    }

logreg_results = {}

for thr, data in pca_datasets.items():
    X_train_pca = data["X_train"]
    X_test_pca = data["X_test"]
    y_train_local = data["y_train"]
    y_test_local = data["y_test"]

    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    )

    model.fit(X_train_pca, y_train_local)

    y_prob = model.predict_proba(X_test_pca)[:, 1]
    y_pred = np.where(y_prob >= 0.3, 1, -1)

    recall_pos1 = recall_score(y_test_local, y_pred, pos_label=1)

    logreg_results[thr] = {
        "model": model,
        "recall_pos1": recall_pos1
    }

    print(f"\n=== Logistic Regression | PCA threshold: {thr:.2f} ===")
    print(f"Recall for label +1 at threshold 0.3: {recall_pos1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_local, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test_local, y_pred))


=== Logistic Regression | PCA threshold: 0.80 ===
Recall for label +1 at threshold 0.3: 0.5238

Confusion Matrix:
[[175 118]
 [ 10  11]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.95      0.60      0.73       293
           1       0.09      0.52      0.15        21

    accuracy                           0.59       314
   macro avg       0.52      0.56      0.44       314
weighted avg       0.89      0.59      0.69       314


=== Logistic Regression | PCA threshold: 0.90 ===
Recall for label +1 at threshold 0.3: 0.3333

Confusion Matrix:
[[215  78]
 [ 14   7]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      0.73      0.82       293
           1       0.08      0.33      0.13        21

    accuracy                           0.71       314
   macro avg       0.51      0.53      0.48       314
weighted avg       0.88      0.71      0.78       314


=== Logistic Regressi

In [5]:
from imblearn.over_sampling import SMOTE
import numpy as np

smote = SMOTE(random_state=42, k_neighbors=5)

smote_results = {}

for thr, data in pca_datasets.items():
    X_train_pca = data["X_train"]
    X_test_pca = data["X_test"]
    y_train_local = data["y_train"]
    y_test_local = data["y_test"]

    # Oversample the minority class in the training set only
    X_resampled, y_resampled = smote.fit_resample(X_train_pca, y_train_local)

    model = LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        random_state=42
    )

    model.fit(X_resampled, y_resampled)

    y_prob = model.predict_proba(X_test_pca)[:, 1]
    y_pred = np.where(y_prob >= 0.3, 1, -1)

    recall_pos1 = recall_score(y_test_local, y_pred, pos_label=1)

    smote_results[thr] = {
        "model": model,
        "recall_pos1": recall_pos1
    }

    print(f"\n=== Logistic Regression + SMOTE | PCA threshold: {thr:.2f} ===")
    print(f"Recall for label +1 at threshold 0.3: {recall_pos1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_local, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test_local, y_pred, zero_division=0))


=== Logistic Regression + SMOTE | PCA threshold: 0.80 ===
Recall for label +1 at threshold 0.3: 0.4286

Confusion Matrix:
[[185 108]
 [ 12   9]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      0.63      0.76       293
           1       0.08      0.43      0.13        21

    accuracy                           0.62       314
   macro avg       0.51      0.53      0.44       314
weighted avg       0.88      0.62      0.71       314


=== Logistic Regression + SMOTE | PCA threshold: 0.90 ===
Recall for label +1 at threshold 0.3: 0.2381

Confusion Matrix:
[[214  79]
 [ 16   5]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.93      0.73      0.82       293
           1       0.06      0.24      0.10        21

    accuracy                           0.70       314
   macro avg       0.49      0.48      0.46       314
weighted avg       0.87      0.70      0.77       314


=== L

### Why SMOTE Didn't Help (and Made Recall Slightly Worse)

1. **Too few real defect anchors** — only ~83 real defect samples in training; SMOTE just interpolates repeatedly between the same small cluster instead of adding new information.

2. **Defects likely span multiple failure modes** — interpolating between two different root-cause failures creates synthetic points that don't match any real defect pattern, blurring the decision boundary.

3. **`class_weight='balanced'` was already doing the job** — it directly reweights the loss function toward catching defects; once SMOTE balanced the classes, this weighting became a no-op, so SMOTE effectively replaced a stronger signal with a weaker one.

4. **High dimensionality weakens "nearest neighbor"** — even at 88 PCA components, distances between points become less meaningful, making SMOTE's synthetic points less trustworthy.

5. **Conclusion** — with a small, high-dimensional, multi-mode minority class, SMOTE added noise instead of signal. This is a known, reportable limitation of SMOTE, not an implementation mistake.